# Khan Academy Analytics Assignment

## Objective
Build a scalable pipeline to track student reading fluency across schools.

In [ ]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('khan_academy.db')

## Load Data

In [ ]:
student_df = pd.read_json('StudentDetails.json')
cmf_input_df = pd.read_json('CMFInput.json')
cmf_metrics_df = pd.read_json('CMFMetrics.json')

## Store in SQLite

In [ ]:
student_df.to_sql('student_details', conn, if_exists='replace', index=False)
cmf_input_df.to_sql('cmf_input', conn, if_exists='replace', index=False)
cmf_metrics_df.to_sql('cmf_metrics', conn, if_exists='replace', index=False)

## Create Final Dataset

In [ ]:
query = '''
SELECT 
    s.id AS student_id,
    s.name,
    s.school_id,
    s.grade,
    ci.id AS attempt_id,
    ci.submitted_at,
    cm.wpm,
    cm.wcpm,
    cm.pronunciation,
    cm.fluency,
    cm.noise
FROM student_details s
LEFT JOIN cmf_input ci ON s.id = ci.child_id
LEFT JOIN cmf_metrics cm ON ci.id = cm.input_id
'''

df = pd.read_sql(query, conn)
df.head()

## Latest Attempt

In [ ]:
df['submitted_at'] = pd.to_datetime(df['submitted_at'])
df = df.sort_values(['student_id', 'submitted_at'], ascending=[True, False])
latest_df = df.drop_duplicates('student_id')
latest_df.head()

## School Level Tracker

In [ ]:
school_tracker = latest_df.groupby('school_id').agg(
    total_students=('student_id', 'nunique'),
    avg_wcpm=('wcpm', 'mean'),
    avg_wpm=('wpm', 'mean'),
    avg_fluency=('fluency', 'mean'),
    avg_pronunciation=('pronunciation', 'mean')
).reset_index()

school_tracker

In [ ]:
school_tracker.to_csv('school_tracker.csv', index=False)

## School Specific Reports

In [ ]:
report_065 = latest_df[latest_df['school_id'] == 'SCH_134065']
report_141 = latest_df[latest_df['school_id'] == 'SCH_134141']

report_065.to_csv('SCH_134065_report.csv', index=False)
report_141.to_csv('SCH_134141_report.csv', index=False)

report_065.head()